In [1]:
!pip install open_clip_torch rics pandas pillow annoy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 14.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp312-cp312-linux_x86_64.whl size=551813 sha256=94545cf8f3e7173ea6c3b93eab715550ad36e47c5019b3adf3ba9e3d75570a84
  Stored in directory: /root/.cache/pip/wheels/db/b9/53/a3b2d1fe1743abadddec6aa541294b24fdbc39d7800bc57311
Successfully built annoy


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import pandas as pd
import torch
from PIL import Image
import open_clip
from annoy import AnnoyIndex
import os
import json
from tqdm import tqdm

# 1. Configuració de rutes
PATH_CSV = '/content/drive/MyDrive/TFM-Sara/support_data/input.csv'
PATH_FOTOS = '/content/drive/MyDrive/TFM-Sara/support_data/images/'
PATH_OUTPUT_ANN = '/content/drive/MyDrive/TFM-Sara/support_data/index/support_database.ann'
PATH_OUTPUT_METADATA = '/content/drive/MyDrive/TFM-Sara/support_data/index/metadata_index.json'

# 2. Càrrega del Dataset
# Nota: Comprova si el teu CSV usa coma (,) o punt i coma (;) com a separador
try:
    # Intentar coma primero (formato estándar), luego punto y coma
    try:
        df = pd.read_csv(PATH_CSV, sep=',', encoding='utf-8')
        if df.shape[1] < 2:  # Si solo hay una columna, probablemente es ';'
            raise ValueError('Separador incorrecto')
    except (ValueError, KeyError):
        df = pd.read_csv(PATH_CSV, sep=';', encoding='utf-8')
except Exception as e:
    print(f"Error llegint el CSV: {e}")

# Netegem files sense any o sense nom de fitxer
df = df.dropna(subset=['year', 'filename'])

# 3. Carregar Model OpenCLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model = model.to(device)
model.eval()

# 4. Generació d'Embeddings i Índex
dimension = 512
annoy_index = AnnoyIndex(dimension, 'angular')
metadata_map = {}

# Pre-calcular archivos existentes una sola vez (evita N llamadas os.path.exists)
existing_files = set(os.listdir(PATH_FOTOS))

# Filtrar filas válidas del CSV
valid_rows = [
    (str(row['filename']).strip(), row)
    for _, row in df.iterrows()
    if str(row['filename']).strip() in existing_files
]
print(f"Processant {len(valid_rows)} fotos vàlides de {len(df)} al CSV")
print(f"Fotos no trobades al disc: {len(df) - len(valid_rows)}")

# Batch inference — BATCH_SIZE=32 óptimo para T4; reducir si hay OOM
BATCH_SIZE = 32
contador_exit = 0

for batch_start in tqdm(range(0, len(valid_rows), BATCH_SIZE),
                        desc="Batches OpenCLIP"):
    batch = valid_rows[batch_start:batch_start + BATCH_SIZE]
    batch_tensors, batch_meta = [], []

    for img_name, row in batch:
        img_path = os.path.join(PATH_FOTOS, img_name)
        try:
            tensor = preprocess(Image.open(img_path))
            batch_tensors.append(tensor)
            batch_meta.append((img_name, row))
        except Exception as e:
            print(f"Error llegint {img_name}: {e}")

    if not batch_tensors:
        continue

    # 1 llamada GPU por batch en lugar de N
    batch_input = torch.stack(batch_tensors).to(device)
    with torch.no_grad():
        features = model.encode_image(batch_input)
        features /= features.norm(dim=-1, keepdim=True)
    vectors = features.cpu().numpy()

    for vector, (img_name, row) in zip(vectors, batch_meta):
        annoy_index.add_item(contador_exit, vector)
        metadata_map[contador_exit] = {
            "year": int(row['year']),
            "fons": str(row['nom_fons']),
            "caption": str(row['caption']),
            "lloc": str(row['toponims']),
            "noms": str(row['noms_propis']),
            "filename": img_name
        }
        contador_exit += 1

# 5. Guardar resultats
if contador_exit > 0:
    annoy_index.build(50, n_jobs=-1)  # n_jobs=-1 usa todos los cores disponibles
    annoy_index.save(PATH_OUTPUT_ANN)

    with open(PATH_OUTPUT_METADATA, 'w', encoding='utf-8') as f:
        json.dump(metadata_map, f, ensure_ascii=False, indent=4)

    print(f"\n✅ Fet! S'han indexat {contador_exit} fotos.")
    print(f"Arxius generats: {PATH_OUTPUT_ANN} i {PATH_OUTPUT_METADATA}")
else:
    print("\n❌ Error: No s'ha pogut indexar cap imatge. Revisa que els noms al CSV coincideixin amb els fitxers de la carpeta.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Processant fotos des de: /content/drive/MyDrive/TFM-Sara/support_data/images/


100%|██████████| 312/312 [03:59<00:00,  1.30it/s]



✅ Fet! S'han indexat 312 fotos.
Arxius generats: /content/drive/MyDrive/TFM-Sara/support_data/index/support_database.ann i /content/drive/MyDrive/TFM-Sara/support_data/index/metadata_index.json
